In [ ]:
from pymoo.problems import get_problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import differential_evolution

In [ ]:
# this is a dummy problem - we can change it later
problem = get_problem('dtlz2',n_var=2,n_obj=2)

# Approach 1 - Obtain solutions first, and then apply HIVES method

In [ ]:
# apply an evolutionary algorithm to get Alternatives or solutions. Lets make a rule of thumb, if the number of objectives <4, 
# we apply NSGA-II, otherwise NSGA-III

algorithm = NSGA2(pop_size=100)

res = minimize(problem,
               algorithm,
               ('n_gen', 100),
               seed=1,
               verbose=True)

In [ ]:
# these are the final results in both objective and decision spaces
X_opt = res.X
F_opt = res.F

In [ ]:
# if size F <4, we use scatter plot, otherwise parallel coordinate plot
plt.scatter(F_opt[:,0],F_opt[:,1])
plt.xlabel('$f_1$')
plt.ylabel('$f_2$')
plt.show()

In [ ]:
# We now rank these solutions based on HIVES method

# Approach 2 - Apply HIVES method first and then do an optimisation based on the resulting weights

In [ ]:
# lets imagine, we get the final weight vector using HIVES method - denoted as gamma_j' (equation 17) in the paper
gamma_j_prime = [25, 75] # for two objectives
# We can then use a single objective optimiser to get the final solution given the original objective function

In [ ]:
# lets define a scalarising function
def TCH(x,weights,z_ideal,z_nadir):
    f_value = problem.evaluate(x) #coming from original objective functions
    f_value = (f_value - z_ideal)/(z_nadir - z_ideal) # for normalisation
    return np.max(weights*np.absolute(f_value))

In [ ]:
bounds =[(0,1),(0,1)] # these are the bounds for decision variable values
z_ideal = np.array([0,0]) # ideal objective vector for the given optimisation problem
z_nadir = np.array([1,1]) # nadir objective vector for the given optimisation problem
weights = gamma_j_prime
results = differential_evolution(TCH,bounds=bounds,args=(weights,z_ideal,z_nadir))

In [ ]:
# Final solution for the given problem using the weights from HIVES method is
x_opt = results.x
f_opt = problem.evaluate(x_opt)
print(x_opt,f_opt)